In [1]:
import enum
import os

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from torch.nn.functional import softmax
from torch.optim import AdamW
from torch.utils.data import DataLoader

from internal.data_types import HistologyDataset
from internal.nn.model import train_one_epoch, validate
from internal.nn.test_time_augmentation import apply_tta
from internal.nn.weighted_random_sampler import make_weighted_sampler
from internal.persistence_manager import PersistenceManager

data = PersistenceManager.load_dataset()
test_df = data.test_df
train_df = data.train_df
train_transforms = data.train_transforms
val_test_transforms = data.val_test_transforms
idx2label = data.idx2label

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cuda_is_available = torch.cuda.is_available()
print(f'Using device: {device}')

Arrays and scalers loaded successfully from: /home/andre/university/AN2DL-Challenge-2/notebooks/processed/dataset.joblib
Using device: cuda


In [2]:
class PreTrainedArchitectures(enum.Enum):
    EFFICIENTNETV2_S = "tf_efficientnetv2_s.in21k"
    CONVNEXT_TINY = "convnext_tiny"

MODEL_TO_USE: PreTrainedArchitectures = PreTrainedArchitectures.CONVNEXT_TINY

# tf_efficientnetv2_s.in21k

In [3]:
if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNETV2_S:
    N_FOLDS = data.num_K_folds
    IMAGE_SIZE = data.image_size
    BATCH_SIZE = 4
    PRETRAINED_MODEL = MODEL_TO_USE.value
    N_CLASSES = 4 # number of classes in the dataset (labels)
    N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4
    EPOCHS_STAGE1 = 8
    EPOCHS_STAGE2 = 12

    for fold in range(N_FOLDS):
        print(f"\n========== Fold {fold} ==========")

        train_df_split = train_df[train_df["fold"] != fold].reset_index(drop=True)
        val_df_split   = train_df[train_df["fold"] == fold].reset_index(drop=True)

        train_dataset = HistologyDataset(train_df_split, transforms=train_transforms, is_train=True, image_size=IMAGE_SIZE)
        val_dataset   = HistologyDataset(val_df_split,   transforms=val_test_transforms, is_train=True, image_size=IMAGE_SIZE)

        sampler = make_weighted_sampler(train_df_split)
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                                  shuffle=False, num_workers=N_WORKERS, pin_memory=cuda_is_available)

        # --- create fresh model for this fold ---
        model = timm.create_model(
            PRETRAINED_MODEL,
            pretrained=True,
            num_classes=N_CLASSES
        ).to(device)

        # --- Stage 1: freeze backbone, train classifier head ---
        print("\n--- Stage 1: Training classifier head ---")

        # --- 1.1. freeze feature extractor layers ---
        for param in model.parameters():
            param.requires_grad = False

        # 2) unfreeze classifier head (EffNetV2 uses .classifier)
        for param in model.classifier.parameters():
            param.requires_grad = True

        # --- 1.2. define loss, optimizer, scheduler ---
        criterion = nn.CrossEntropyLoss()
        head_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = AdamW(head_params, lr=1e-3, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE1)

        # --- 1.3. train for several epochs ---
        best_f1 = 0.0
        best_state = None
        for epoch in range(1, EPOCHS_STAGE1+1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE1}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = model.state_dict().copy()
                torch.save(best_state, f"best_effv2_stage1_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)
            print(f"Restored best Stage 1 weights for fold {fold} (F1={best_f1:.4f})")

        # --- Stage 2: unfreeze whole model, fine-tune ---
        print("\n--- Stage 2: Fine-tuning entire model ---")

        # --- 2.1. unfreeze entire model ---
        for param in model.parameters():
            param.requires_grad = True

        # --- 2.2. define loss, optimizer, scheduler ---
        optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE2)

        # --- 2.3. mild class weights ---
        class_counts = torch.tensor([445, 414, 397, 156], dtype=torch.float32) # 445 LumB, 414 LumA, 397 Her2, 156 TN
        class_weights = (class_counts.sum() / class_counts)
        class_weights = class_weights / class_weights.mean()
        # criterion = FocalLoss(alpha=class_weights, gamma=2.0)
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)

        # --- 2.4. train for several epochs ---
        best_f1 = 0.0
        best_state = None

        for epoch in range(1, EPOCHS_STAGE2 + 1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE2}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = model.state_dict().copy()
                torch.save(best_state, f"best_effv2_stage2_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)   # restore best val-F1 weights

        # --- save model for this fold ---
        torch.save(model.state_dict(), f"effv2_s_fold{fold}.pth")

# convnext_tiny

In [4]:
if MODEL_TO_USE == PreTrainedArchitectures.CONVNEXT_TINY:
    N_FOLDS = data.num_K_folds
    IMAGE_SIZE = data.image_size
    BATCH_SIZE = 4
    PRETRAINED_MODEL = MODEL_TO_USE.value
    N_CLASSES = 4  # number of classes in the dataset (labels)
    N_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 4
    EPOCHS_STAGE1 = 8
    EPOCHS_STAGE2 = 12

    for fold in range(N_FOLDS):
        print(f"\n========== Fold {fold} ==========")

        train_df_split = train_df[train_df["fold"] != fold].reset_index(drop=True)
        val_df_split   = train_df[train_df["fold"] == fold].reset_index(drop=True)

        train_dataset = HistologyDataset(
            train_df_split,
            transforms=train_transforms,
            is_train=True,
            image_size=IMAGE_SIZE
        )
        val_dataset = HistologyDataset(
            val_df_split,
            transforms=val_test_transforms,
            is_train=True,
            image_size=IMAGE_SIZE
        )

        sampler = make_weighted_sampler(train_df_split)
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=N_WORKERS,
            pin_memory=cuda_is_available
        )

        # --- create fresh model for this fold ---
        model = timm.create_model(
            PRETRAINED_MODEL,
            pretrained=True,
            num_classes=N_CLASSES
        ).to(device)

        # --- Stage 1: freeze backbone, train classifier HEAD (ConvNeXt) ---
        print("\n--- Stage 1: Training classifier head (ConvNeXt-Tiny) ---")

        # --- 1.1. freeze feature extractor layers ---
        for p in model.parameters():
            p.requires_grad = False

        # --- 1.1. unfreeze only the classifier head (ConvNeXt uses .head) ---
        for p in model.head.parameters():
            p.requires_grad = True

        # --- 1.2. define loss, optimizer, scheduler ---
        criterion = nn.CrossEntropyLoss()
        head_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = AdamW(head_params, lr=1e-3, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE1)

        # --- 1.3. train for several epochs ---
        best_f1 = 0.0
        best_state = None
        for epoch in range(1, EPOCHS_STAGE1 + 1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE1}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = model.state_dict().copy()
                torch.save(best_state, f"best_convnext_stage1_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)
            print(f"Restored best Stage 1 weights for fold {fold} (F1={best_f1:.4f})")

        # --- Stage 2: unfreeze whole model, fine-tune ---
        print("\n--- Stage 2: Fine-tuning entire model (ConvNeXt-Tiny) ---")

        # --- 2.1. unfreeze entire model ---
        for p in model.parameters():
            p.requires_grad = True

        # --- 2.2. define loss, optimizer, scheduler ---
        optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE2)

        # --- 2.3. mild class weights ---
        class_counts = torch.tensor([445, 414, 397, 156], dtype=torch.float32)  # 445 LumB, 414 LumA, 397 Her2, 156 TN
        class_weights = (class_counts.sum() / class_counts)
        class_weights = class_weights / class_weights.mean()
        # criterion = FocalLoss(alpha=class_weights, gamma=2.0)
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)

        # --- 2.4. train for several epochs ---
        best_f1 = 0.0
        best_state = None
        for epoch in range(1, EPOCHS_STAGE2 + 1):
            print(f"\nEpoch {epoch}/{EPOCHS_STAGE2}")
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )
            val_loss, val_acc, val_f1 = validate(
                model, val_loader, criterion, device
            )
            scheduler.step()
            print(
                f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
                f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
            )
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = model.state_dict().copy()
                torch.save(best_state, f"best_convnext_stage2_fold{fold}_f1_{val_f1:.4f}.pth")
                print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")

        if best_state is not None:
            model.load_state_dict(best_state)  # restore best val-F1 weights

        # --- save model for this fold ---
        torch.save(model.state_dict(), f"convnext_tiny_fold{fold}.pth")


========== Fold 0 ==========

--- Stage 1: Training classifier head (ConvNeXt-Tiny) ---

Epoch 1/8


    t_loss=1.4178 | F1(macro)=0.3221 | Acc=0.3276


Confusion matrix:
 [[23  7  3  8]
 [12  7  3 10]
 [14  1  2 13]
 [ 8  2  0  4]]
Train  loss=1.4178 acc=0.3276 f1=0.3221 | Val loss=1.5004 acc=0.3077 f1=0.2559
  🔥 New best F1: 0.2559 – model saved.

Epoch 2/8


    t_loss=1.3144 | F1(macro)=0.3676 | Acc=0.3707


Confusion matrix:
 [[19 13  0  9]
 [14 10  0  8]
 [14  8  0  8]
 [ 8  3  0  3]]
Train  loss=1.3144 acc=0.3707 f1=0.3676 | Val loss=1.5715 acc=0.2735 f1=0.2104

Epoch 3/8


...

Confusion matrix:
 [[ 0  0  0 41]
 [ 0  2  0 30]
 [ 0  1  0 29]
 [ 0  0  0 13]]
Train  loss=1.3584 acc=0.2387 f1=0.1274 | Val loss=1.4463 acc=0.1293 f1=0.0802
  🔥 New best F1: 0.0802 – model saved.

Epoch 11/12


    t_loss=1.3490 | F1(macro)=0.1713 | Acc=0.2624


Confusion matrix:
 [[ 0  0  0 41]
 [ 0  2  0 30]
 [ 0  0  0 30]
 [ 0  0  0 13]]
Train  loss=1.3490 acc=0.2624 f1=0.1713 | Val loss=1.4368 acc=0.1293 f1=0.0806
  🔥 New best F1: 0.0806 – model saved.

Epoch 12/12


    t_loss=1.3093 | F1(macro)=0.1474 | Acc=0.2710


Confusion matrix:
 [[ 0 19  0 22]
 [ 0 14  0 18]
 [ 0  6  0 24]
 [ 0  3  0 10]]
Train  loss=1.3093 acc=0.2710 f1=0.1474 | Val loss=1.4292 acc=0.2069 f1=0.1521
  🔥 New best F1: 0.1521 – model saved.


In [5]:
prefix_filename = "effv2_s" if MODEL_TO_USE == PreTrainedArchitectures.EFFICIENTNETV2_S else "convnext_tiny"

test_dataset = HistologyDataset(
    test_df,
    transforms=val_test_transforms,
    is_train=False,   # returns (img, sample_index)
    image_size=IMAGE_SIZE
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=N_WORKERS,
    pin_memory=cuda_is_available
)

all_fold_probs = []   # list of arrays [N, num_classes]
all_sample_indices = None

for fold in range(N_FOLDS):
    print(f"Inference with fold {fold} model")

    # recreate model and load weights
    model = timm.create_model(
        PRETRAINED_MODEL,
        pretrained=False,
        num_classes=N_CLASSES
    ).to(device)
    state = torch.load(f"{prefix_filename}_fold{fold}.pth", map_location=device)
    model.load_state_dict(state)
    model.eval()

    fold_probs = []
    sample_indices_list = []

    with torch.no_grad():
        for imgs, sample_indices in test_loader:
            imgs = imgs.to(device, non_blocking=True)

            logits = model(imgs)               # [B, num_classes]
            probs = softmax(logits, dim=1)     # [B, num_classes]
            fold_probs.append(probs.cpu().numpy())

            # collect sample indices only once
            if all_sample_indices is None:
                sample_indices_list.extend(sample_indices)

    fold_probs = np.concatenate(fold_probs, axis=0)  # [N, num_classes]
    all_fold_probs.append(fold_probs)

    if all_sample_indices is None:
        all_sample_indices = sample_indices_list

# average probabilities across folds
mean_probs = np.mean(all_fold_probs, axis=0)   # [N, num_classes]
pred_indices = mean_probs.argmax(axis=1)

pred_labels = [idx2label[int(i)] for i in pred_indices]
sample_index_with_ext = [
    f"{si}.png" if not si.endswith(".png") else si
    for si in all_sample_indices
]

submission_df = pd.DataFrame({
    "sample_index": sample_index_with_ext,
    "label": pred_labels
})

submission_df.to_csv("submission_5fold_no_tta.csv", index=False)
print("Saved submission_5fold_no_tta.csv")
print(submission_df.head())


Inference with fold 0 model
Inference with fold 1 model
Inference with fold 2 model
Inference with fold 3 model
Inference with fold 4 model
Saved submission_5fold_no_tta.csv
   sample_index            label
0  img_0000.png        Luminal A
1  img_0001.png        Luminal A
2  img_0002.png        Luminal A
3  img_0003.png  Triple negative
4  img_0004.png        Luminal A


In [7]:
########################################################
# ===== Inference with TTA and 5-Fold Ensembling ===== #
########################################################
all_fold_probs = []
all_sample_indices = None

test_dataset = HistologyDataset(test_df, transforms=val_test_transforms, is_train=False, image_size=IMAGE_SIZE)
test_loader = DataLoader(test_dataset, batch_size=1,  # IMPORTANT: batch_size=1 for per-image TTA
                         shuffle=False, num_workers=N_WORKERS, pin_memory=cuda_is_available)

for fold in range(N_FOLDS):
    print(f"Inference with fold {fold} model")
    model = timm.create_model(PRETRAINED_MODEL, pretrained=False, num_classes=N_CLASSES).to(device)
    model.load_state_dict(torch.load(f"{prefix_filename}_fold{fold}.pth", map_location=device))
    model.eval()

    fold_probs = []
    sample_indices_list = []

    with torch.no_grad():
        for img_tensor, sample_idx in test_loader:
            img_tensor = img_tensor.squeeze(0)  # [3,H,W]
            img_tensor = img_tensor.to(device)

            # -------- TTA: apply multiple augmented views --------
            tta_tensors = apply_tta(img_tensor)

            # accumulate probability predictions
            probs_sum = 0
            for aug_img in tta_tensors:
                aug_img = aug_img.unsqueeze(0).to(device)  # [1,3,H,W]
                logits = model(aug_img)
                probs = softmax(logits, dim=1)  # [1,4]
                probs_sum += probs[0].cpu().numpy()

            # average across TTA views
            avg_probs = probs_sum / len(tta_tensors)
            fold_probs.append(avg_probs)

            if all_sample_indices is None:
                sample_indices_list.append(sample_idx[0])

    fold_probs = np.vstack(fold_probs)  # [N, 4]
    all_fold_probs.append(fold_probs)

    if all_sample_indices is None:
        all_sample_indices = sample_indices_list

mean_probs = np.mean(all_fold_probs, axis=0)  # [N, 4]
pred_indices = mean_probs.argmax(axis=1)
pred_labels = [idx2label[int(i)] for i in pred_indices]

sample_index_with_ext = [
    f"{si}.png" if not si.endswith(".png") else si
    for si in all_sample_indices
]

submission_df = pd.DataFrame({
    "sample_index": sample_index_with_ext,
    "label": pred_labels
})
submission_df.to_csv("submission_5fold_tta.csv", index=False)

print("Saved submission_5fold_tta.csv")

Inference with fold 0 model
Inference with fold 1 model
Inference with fold 2 model
Inference with fold 3 model
Inference with fold 4 model
Saved submission_5fold_tta.csv


In [ ]:
def predict_loader_with_tta(model, loader, device):
    model.eval()
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for imgs, labels in loader:  # note: here we have labels, not sample_index
            imgs = imgs.squeeze(0).to(device)  # if batch_size=1
            tta_imgs = apply_tta(imgs)         # same apply_tta as for test

            probs_sum = 0
            for aug in tta_imgs:
                aug = aug.unsqueeze(0).to(device)
                logits = model(aug)
                probs = softmax(logits, dim=1)
                probs_sum += probs[0].cpu().numpy()

            avg_probs = probs_sum / len(tta_imgs)
            all_probs.append(avg_probs)
            all_targets.append(labels.item())

    all_probs = np.vstack(all_probs)
    all_targets = np.array(all_targets)
    pred_indices = all_probs.argmax(axis=1)

    macro_f1 = f1_score(all_targets, pred_indices, average="macro")
    return macro_f1

fold_f1s = []

for fold in range(N_FOLDS):
    print(f"OOF eval for fold {fold}")

    # build val_df_split for that fold
    val_df_split = train_df[train_df["fold"] == fold].reset_index(drop=True)
    val_dataset = HistologyDataset(val_df_split, transforms=val_test_transforms, is_train=True, image_size=IMAGE_SIZE)
    val_loader  = DataLoader(val_dataset, batch_size=1, shuffle=False,
                             num_workers=N_WORKERS, pin_memory=cuda_is_available)

    model = timm.create_model(PRETRAINED_MODEL, pretrained=False, num_classes=N_CLASSES).to(device)
    model.load_state_dict(torch.load(f"effv2_s_fold{fold}.pth", map_location=device))

    f1 = predict_loader_with_tta(model, val_loader, device)
    fold_f1s.append(f1)
    print("Fold F1 (OOF, with TTA):", f1)

print("Mean OOF F1:", np.mean(fold_f1s))
